Q1. Embedding a query

In [2]:
from embedder import Embedder

embed = Embedder()

q1 = "How does approximate nearest neighbor search work?"
v = embed.encode(q1)

print(len(v), v[0])

384 -0.02058203437252893


Q2. Cosine similarity

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [9]:
import numpy as np

q1 = "How does approximate nearest neighbor search work?"
v_query = np.array(embed.encode(q1))

target_path = "02-vector-search/lessons/07-sqlitesearch-vector.md"
target_doc = None

for doc in documents:
    if doc["filename"] == target_path:
        target_doc = doc
        break
    
if target_doc is None:
    raise ValueError(f"File {target_path} not found.")

v_doc = np.array(embed.encode(target_doc["content"]))

cosine_similarity = np.dot(v_query, v_doc)

print(f"Cosine Similarity: {cosine_similarity}")

Cosine Similarity: 0.36107026789538205


Q3. Chunking and search by hand

In [10]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

X = np.array(embed.encode_batch([chunk["content"] for chunk in chunks]))

scores = X.dot(v)

best_idx = np.argmax(scores)
best_chunk = chunks[best_idx]

print(best_chunk["filename"])

02-vector-search/lessons/07-sqlitesearch-vector.md


Q4. Vector search with minsearch

In [22]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, chunks)

query = "What metric do we use to evaluate a search engine?"
v_query = np.array(embed.encode(query))

results = vindex.search(v_query, num_results=5)

print(results[0]["filename"])

04-evaluation/lessons/05-search-metrics.md


Q5. Text search vs vector search

In [24]:
from minsearch import Index

tindex = Index(text_fields=["content"])
tindex.fit(chunks)

query = "How do I store vectors in PostgreSQL?"

# Text search
text_results = tindex.search(query, num_results=5)
text_filenames = set(r["filename"] for r in text_results)

# Vector search
v_query = np.array(embed.encode(query))
vector_results = vindex.search(v_query, num_results=5)
vector_filenames = set(r["filename"] for r in vector_results)

# Difference
only_in_vector = vector_filenames - text_filenames

print("Text results:", text_filenames)
print("Vector results:", vector_filenames)
print("Only in vector:", only_in_vector)

Text results: {'02-vector-search/lessons/02-embeddings.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md'}
Vector results: {'02-vector-search/lessons/08-pgvector.md', '03-orchestration/lessons/05-rag.md'}
Only in vector: {'02-vector-search/lessons/08-pgvector.md'}


Q6. Hybrid search

In [27]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [29]:
from minsearch import Index

tindex = Index(text_fields=["content"])
tindex.fit(chunks)

query = "How do I give the model access to tools?"

# Text search
text_results = tindex.search(query, num_results=5)

# Vector search
v_query = np.array(embed.encode(query))
vector_results = vindex.search(v_query, num_results=5)

results = rrf([vector_results, text_results])

print(results[0]["filename"])

01-agentic-rag/lessons/13-function-calling.md
